In [1]:
import sys, os, numpy as np, pandas as pd, torch, torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import models
from sklearn.metrics import confusion_matrix, f1_score, accuracy_score, classification_report
import matplotlib.pyplot as plt

sys.path.append("../src")
from dataset import gather_samples, stratified_split, RiceLeafDataset, CLASSES

os.makedirs("../figures", exist_ok=True)
os.makedirs("../tables", exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device, CLASSES)

cuda ['Healthy', 'Insect', 'Leaf Scald', 'Rice Blast', 'Rice Leaffolder', 'Rice Stripes', 'Rice Tungro']


In [2]:
samples = gather_samples()
train_s, val_s, test_s = stratified_split(samples)
test_set = RiceLeafDataset(test_s, split='test')
test_loader = DataLoader(test_set, batch_size=32, shuffle=False, num_workers=2)
print(len(samples), len(train_s), len(val_s), len(test_set))

2753 1927 413 413


In [3]:
def build_model(name, n=7):
    if name == "resnet50":
        m = models.resnet50(weights=None); m.fc = nn.Linear(m.fc.in_features, n)
    elif name == "efficientnet_b0":
        m = models.efficientnet_b0(weights=None); m.classifier[1] = nn.Linear(m.classifier[1].in_features, n)
    elif name == "mobilenetv3_large":
        m = models.mobilenet_v3_large(weights=None); m.classifier[3] = nn.Linear(m.classifier[3].in_features, n)
    elif name == "convnext_tiny":
        m = models.convnext_tiny(weights=None); m.classifier[2] = nn.Linear(m.classifier[2].in_features, n)
    elif name == "swin_tiny":
        m = models.swin_t(weights=None); m.head = nn.Linear(m.head.in_features, n)
    return m

NAMES = {"resnet50": "ResNet50", "efficientnet_b0": "EfficientNet-B0",
         "mobilenetv3_large": "MobileNetV3-Large", "convnext_tiny": "ConvNeXt-Tiny",
         "swin_tiny": "Swin-Tiny"}

@torch.no_grad()
def predict(model):
    model.eval(); ys, ps = [], []
    for x, y in test_loader:
        ps.append(model(x.to(device)).argmax(1).cpu().numpy()); ys.append(np.asarray(y))
    return np.concatenate(ys), np.concatenate(ps)

preds, rows = {}, []
for key, disp in NAMES.items():
    m = build_model(key)
    m.load_state_dict(torch.load(f"../notebooks/{key}_best.pth", map_location="cpu"))
    m.to(device)
    y, p = predict(m)
    preds[key] = (y, p)
    rows.append({"Model": disp,
                 "Params (M)": round(sum(q.numel() for q in m.parameters())/1e6, 2),
                 "Accuracy": round(accuracy_score(y, p), 4),
                 "Macro-F1": round(f1_score(y, p, average="macro"), 4)})
    del m; torch.cuda.empty_cache()

comp = pd.DataFrame(rows)
inf_ms = {"ResNet50": np.nan, "EfficientNet-B0": 8.44, "MobileNetV3-Large": 7.60,
          "ConvNeXt-Tiny": 9.63, "Swin-Tiny": 12.90}
comp["Inference (ms)"] = comp["Model"].map(inf_ms)
print(comp)

y, p = preds["convnext_tiny"]
print(classification_report(y, p, target_names=CLASSES, digits=3))

               Model  Params (M)  Accuracy  Macro-F1  Inference (ms)
0           ResNet50       23.52    0.7627    0.7346             NaN
1    EfficientNet-B0        4.02    0.7918    0.7483            8.44
2  MobileNetV3-Large        4.21    0.7530    0.7234            7.60
3      ConvNeXt-Tiny       27.83    0.7676    0.7215            9.63
4          Swin-Tiny       27.52    0.7918    0.7682           12.90
                 precision    recall  f1-score   support

        Healthy      0.840     0.878     0.859        90
         Insect      0.810     0.810     0.810        42
     Leaf Scald      0.745     0.651     0.695        63
     Rice Blast      0.939     0.886     0.912       105
Rice Leaffolder      0.667     0.757     0.709        37
   Rice Stripes      0.459     0.425     0.442        40
    Rice Tungro      0.568     0.694     0.625        36

       accuracy                          0.768       413
      macro avg      0.718     0.729     0.721       413
   weighted av